# 13 Testing optimized Models by LL

## Import

In [7]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

import matplotlib as mpl
import matplotlib.pyplot as plt

In [8]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [9]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [10]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

idx_train_val = np.concatenate([idx_train, idx_val])
df_train_val = df_raw.iloc[idx_train_val]
x_train_val, y_train_val = df_train_val[feat_cols], df_train_val["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)],
        "train_val": [len(df_train_val), len(x_train_val), len(y_train_val)]
    }
)

train        [476168, 476168, 476168]
val             [59522, 59522, 59522]
test            [59522, 59522, 59522]
full         [595212, 595212, 595212]
train_val    [535690, 535690, 535690]
dtype: object

## Hilfsvariablen

In [30]:
results = []
train_times = {}

In [12]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [13]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_test": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

In [14]:
def mv_for_cb(df, fill_var="missing"):
    """df copy mit ersetzten nans"""
    outs = df.copy()
    for col in outs.columns:
        if outs[col].dtype.name == "category":
            if fill_var not in outs[col].cat.categories:
                outs[col] = outs[col].cat.add_categories([fill_var])
            outs[col] = outs[col].fillna(fill_var)
    return outs

In [15]:
x_train_cb = mv_for_cb(x_train)
x_val_cb = mv_for_cb(x_val)
x_test_cb = mv_for_cb(x_test)
x_full_cb = mv_for_cb(x_full)
x_train_val_cb = mv_for_cb(x_train_val)

x_train_no_calc_cb = x_train_cb[feat_cols_no_calc]
x_val_no_calc_cb = x_val_cb[feat_cols_no_calc]
x_test_no_calc_cb = x_test_cb[feat_cols_no_calc]
x_full_no_calc_cb = x_full_cb[feat_cols_no_calc]
x_train_val_no_calc_cb = x_train_val_cb[feat_cols_no_calc]

pd.Series(
    {
        "train": [len(x_train_cb), len(x_train_no_calc_cb)],
        "val": [len(x_val_cb), len(x_val_no_calc_cb)],
        "test": [len(x_test_cb), len(x_test_no_calc_cb)],
        "full": [len(x_full_cb), len(x_full_no_calc_cb)],
        "train_val": [len(x_train_val_cb), len(x_train_val_no_calc_cb)]
    }
)

train        [476168, 476168]
val            [59522, 59522]
test           [59522, 59522]
full         [595212, 595212]
train_val    [535690, 535690]
dtype: object

In [16]:
def load_cbm(name):
    """cbm modell laden"""
    modell = CatBoostClassifier()
    modell.load_model(f"{name}.cbm")
    return modell

## Logreg
Nicht gespeichert, deswegen nochmal training mit:

|Parameter|Wert|
|---|---|
|C|0.01|
|class_weight|balanced|
|fit_intercept|False|
|penalty|None|
|solver|newton-cg|
|tol|0.01|

In [17]:
L_logreg_test_t = make_pipeline(
    make_column_transformer(
            (
                make_pipeline(
                    SimpleImputer(strategy="median", add_indicator=True),
                    StandardScaler()
                ),
                num_cols_no_calc
            ),
            (
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"),
                low_kard_cols
            ),
            (
                TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE),
                high_kard_cols
            ),
            (
                "passthrough",
                bin_cols_no_calc
            )
        ),
    LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        C=0.01,
        class_weight="balanced",
        fit_intercept=False,
        penalty=None,
        solver="newton-cg",
        tol=0.01
        )
)

In [31]:
name = "L_logreg_test_t"

start = time.time()
L_logreg_test_t.fit(x_train, y_train)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_logreg_test_t, x_train, y_train, x_test, y_test, train_times[name], best_iter=None)
)

c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\linear_model\_logistic.py:1443: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  wa

In [19]:
L_logreg_test_tv = make_pipeline(
    make_column_transformer(
            (
                make_pipeline(
                    SimpleImputer(strategy="median", add_indicator=True),
                    StandardScaler()
                ),
                num_cols_no_calc
            ),
            (
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"),
                low_kard_cols
            ),
            (
                TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE),
                high_kard_cols
            ),
            (
                "passthrough",
                bin_cols_no_calc
            )
        ),
    LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        C=0.01,
        class_weight="balanced",
        fit_intercept=False,
        penalty=None,
        solver="newton-cg",
        tol=0.01
        )
)

In [32]:
name = "L_logreg_test_tv"

start = time.time()
L_logreg_test_tv.fit(x_train_val, y_train_val)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_logreg_test_tv, x_train_val, y_train_val, x_test, y_test, train_times[name], best_iter=None)
)

c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\linear_model\_logistic.py:1443: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  wa

## Catboost gespeichert

In [21]:
L_cb_test_01_t = load_cbm("L_cb_opt_01")
L_cb_test_02_t = load_cbm("L_cb_opt_02")
L_cb_test_03_t = load_cbm("L_cb_opt_03")

In [33]:
results.append(
    eval_modell("L_cb_test_01_t", L_cb_test_01_t, x_train_cb, y_train, x_test_cb, y_test, None, best_iter=L_cb_test_01_t.get_best_iteration())
)
results.append(
    eval_modell("L_cb_test_02_t", L_cb_test_02_t, x_train_no_calc_cb, y_train, x_test_no_calc_cb, y_test, None, best_iter=L_cb_test_02_t.get_best_iteration())
)
results.append(
    eval_modell("L_cb_test_03_t", L_cb_test_03_t, x_train_no_calc_cb, y_train, x_test_no_calc_cb, y_test, None, best_iter=L_cb_test_03_t.get_best_iteration())
)

In [23]:
fixed_params_plain = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Plain"
}

In [24]:
fixed_params_ordered = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Ordered"
}

In [25]:
angepasste_params = [
    "learning_rate", 
    "depth", 
    "l2_leaf_reg", 
    "random_strength", 
    "one_hot_max_size", 
    "leaf_estimation_iterations", 
    "auto_class_weights", 
    "bootstrap_type", 
    "bagging_temperature", 
    "subsample"]

In [26]:
{param : wert for param, wert in L_cb_test_01_t.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 105,
 'l2_leaf_reg': 8.593733788,
 'random_strength': 0.01142319106,
 'subsample': 0.9178574681,
 'depth': 7,
 'auto_class_weights': 'Balanced',
 'learning_rate': 0.04761112109,
 'leaf_estimation_iterations': 2,
 'bootstrap_type': 'Bernoulli'}

In [27]:
{param : wert for param, wert in L_cb_test_02_t.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 10,
 'l2_leaf_reg': 3.914318323,
 'random_strength': 0.5015455484,
 'subsample': 0.7854715586,
 'depth': 4,
 'auto_class_weights': 'None',
 'learning_rate': 0.1759581715,
 'leaf_estimation_iterations': 2,
 'bootstrap_type': 'Bernoulli'}

In [28]:
{param : wert for param, wert in L_cb_test_03_t.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 10,
 'l2_leaf_reg': 21.72735786,
 'random_strength': 1.770355582,
 'depth': 9,
 'bagging_temperature': 0.004430230707,
 'auto_class_weights': 'None',
 'learning_rate': 0.07522925735,
 'leaf_estimation_iterations': 3,
 'bootstrap_type': 'Bayesian'}

## Evaluation

In [34]:
df_results = pd.DataFrame(results).sort_values("auc_test", ascending=False)
df_results

,model_idx,auc_train,auc_test,gini,delta_auc,best_iter,trainingszeit
3,L_cb_test_02_t,0.664717,0.647273,0.294546,0.017444,518.0,NaN
4,L_cb_test_03_t,0.655782,0.644887,0.289773,0.010896,412.0,NaN
2,L_cb_test_01_t,0.740788,0.643028,0.286056,0.097760,549.0,NaN
1,L_logreg_test_tv,0.630187,0.633230,0.266460,-0.003043,NaN,4.013533
0,L_logreg_test_t,0.631645,0.633102,0.266203,-0.001457,NaN,3.474550


In [36]:
train_times_nb12 = {
    "L_cb_test_01_t": 24.047395,
    "L_cb_test_02_t": 52.476140,
    "L_cb_test_03_t": 192.045585,
}

for modell,tt in train_times_nb12.items():
    df_results.loc[df_results["model_idx"] == modell, "trainingszeit"] = tt

df_results

,model_idx,auc_train,auc_test,gini,delta_auc,best_iter,trainingszeit
3,L_cb_test_02_t,0.664717,0.647273,0.294546,0.017444,518.0,52.476140
4,L_cb_test_03_t,0.655782,0.644887,0.289773,0.010896,412.0,192.045585
2,L_cb_test_01_t,0.740788,0.643028,0.286056,0.097760,549.0,24.047395
1,L_logreg_test_tv,0.630187,0.633230,0.266460,-0.003043,NaN,4.013533
0,L_logreg_test_t,0.631645,0.633102,0.266203,-0.001457,NaN,3.474550


## Notizen
- Logreg muss ich neu trainieren (dann auch einmal nur auf train und einmal auf train_val)
- Die catboost Modelle habe ich gespeichert (auf train trainiert), diese trainiere ich zusätslich nochmal auf train_val
- bei catboost hatte ich einen denkfehler, das val ja sowieso im training zur validierung verwendet wird, macht es keinen sinn das modell auch auf train_val zu trainieren ohne overfitting auf test zu erzeugen
- bei gespeicherten Modellen hab ich grad keine zeiten
- ich habe in 10 test aus versehen in full übergeben das modell ist nicht mit einzubeziehen.
- Nächster Schritt:
    - PCA ?
    - Laufzeit war in ordnung, überso wochenende kann ich die opti nochmal auf dem ganzen sample laufen lassen.
    - Target Encoding bei CatBoost nochmal anschauen


Ressourcen [Aufrufdatum: 13.08.2026]:
- https://catboost.ai/docs/en/concepts/python-reference_catboost_load_model
